### Setup

In [ ]:
import pandas as pd
import json
import os
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
import ipywidgets as widgets
import torch 
import shutil
from transformers import pipeline
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device : ',device)  

# PATH
CONFIG_PATH = '../config.json'
with open(CONFIG_PATH,'r') as f:
    config = json.load(f)
    
# DATA
IMAGE_SAMP_FOLDER = config.get("IMAGE_SAMP_FOLDER")
LABEL_SAMP_FOLDER = config.get("LABEL_SAMP_FOLDER")
CAPTION_FOLDER = config.get("CAPTION_FOLDER")
ADD_INFO_FOLDER = config.get("ADD_INFO_FOLDER")
INSTRUCTION_FOLDER = config.get("INSTRUCTION_FOLDER")
SEED_LABEL_FOLDER = config.get("SEED_LABEL_FOLDER")
IMAGE_FILE = sorted(os.listdir(IMAGE_SAMP_FOLDER))
LABEL_FILE = sorted(os.listdir(LABEL_SAMP_FOLDER))
COMBINED_IMAGE_FOLDER = config.get("COMBINED_IMAGE_FOLDER")
COMBINED_LABEL_FOLDER = config.get("COMBINED_LABEL_FOLDER")

TRAIN_IMAGE_FOLDER = config.get("TRAIN_IMAGE_FOLDER")
TRAIN_LABEL_FOLDER = config.get("TRAIN_LABEL_FOLDER")
TEST_IMAGE_FOLDER = config.get("TEST_IMAGE_FOLDER")
TEST_LABEL_FOLDER = config.get("TEST_LABEL_FOLDER")
NIKE_LABEL_FOLDER = config.get("NIKE_LABEL_FOLDER")
ZARA_LABEL_FOLDER = config.get("ZARA_LABEL_FOLDER")

# MODEL
SUMMARIZE_MODEL = "facebook/bart-large-cnn"

In [ ]:
print('Augment_KFashion_Image : ', len(os.listdir("../Data/Augmented/Augment_KFashion_Image/")))
print('Augment_CLIP : ', len(os.listdir("../Data/Augmented/Augment_CLIP/")))
print('Combined_Image_KFashion : ', len(os.listdir("../Data/Combined/Combined_Image_KFashion/")))
print('Combined_Label_CLIP : ', len(os.listdir("../Data/Combined/Combined_Label_CLIP/")))
print('Combined_Label_CLIP_Expand : ', len(os.listdir("../Data/Combined/Combined_Label_CLIP_Expand/")))
print('*'*50)
print('Augment_SD_Image : ', len(os.listdir("../Data/Augmented/Augment_SD_Image/")))
print('Augment_SD_Label : ', len(os.listdir("../Data/Augmented/Augment_SD_Label/")))
print('Combined_Image : ', len(os.listdir("../Data/Combined/Combined_Image/")))
print('Combined_Label : ', len(os.listdir("../Data/Combined/Combined_Label/")))
print('Combined_Label_Expand : ', len(os.listdir("../Data/Combined/Combined_Label_Expand/")))

### Generate Caption

- Extract only clothing details from labeling data -> Caption data

In [9]:
def load_label(label_path):
    with open(label_path, 'r', encoding='utf-8') as file:
        label = json.load(file)
    return label

def extract_details(label):
    style = label['데이터셋 정보']['데이터셋 상세설명']['라벨링']['스타일']
    outer = label['데이터셋 정보']['데이터셋 상세설명']['라벨링']['아우터']
    pants = label['데이터셋 정보']['데이터셋 상세설명']['라벨링']['하의']
    dress = label['데이터셋 정보']['데이터셋 상세설명']['라벨링']['원피스']
    top = label['데이터셋 정보']['데이터셋 상세설명']['라벨링']['상의']
    label_data_detail = [style, outer, pants, dress, top]
    return label_data_detail

def create_deatil_caption(label_data_detail):
    caption = []
    if '스타일' in label_data_detail[0][0]:
        caption.append(f"스타일 : {label_data_detail[0][0]['스타일']},")
    if '서브스타일' in label_data_detail[0][0]:
        caption.append(f"서브스타일 : {label_data_detail[0][0]['서브스타일']},")
    outer_detail = label_data_detail[1]
    pant_detail = label_data_detail[2]
    dress_detail = label_data_detail[3]
    top_detail = label_data_detail[4]
    if outer_detail is not [{}]:
        caption.append(f"아우터 : {outer_detail[0]},")
    if pant_detail is not [{}]:
        caption.append(f"하의 : {pant_detail[0]},")
    if dress_detail is not [{}]:
        caption.append(f"원피스 : {dress_detail[0]},")
    if top_detail is not [{}]:
        caption.append(f"상의 : {top_detail[0]},")
    return " ".join(caption)


In [10]:
# Extract clothing description captions for each image
caption_li = []
for label_path in LABEL_FILE:
    label = load_label(LABEL_SAMP_FOLDER + label_path)
    label_data_detail = extract_details(label)
    caption = create_deatil_caption(label_data_detail)
    caption_li.append(caption)
    
## Save captions as JSON files with filenames matching each image number
for label_path, caption in zip(LABEL_FILE, caption_li):
    # Extract file number (e.g., '268') from the label path
    file_number = os.path.basename(label_path).split('.')[0]
    # Define the caption file path
    caption_file_path = os.path.join(CAPTION_FOLDER, f"{file_number}.json")
    # Save the caption as a JSON file
    with open(caption_file_path, 'w', encoding='utf-8') as file:
        json.dump({"caption": caption}, file, ensure_ascii=False, indent=4)

### Generate Add Info

In [11]:
def checking_image_add_info_data(image_path):
    image = Image.open(image_path)
    plt.imshow(image)
    plt.axis('off')
    plt.show()

    # Input field for additional information
    add_info_text = widgets.Text(
        value='',
        placeholder='Enter additional information',
        description='Info:',
        disabled=False
    )
    display(add_info_text)

    # Function to save the text when the button is clicked
    def on_button_click(b):
        add_info_path = os.path.join(ADD_INFO_FOLDER, os.path.basename(image_path).split('.')[0] + '.txt')
        pd.Series([add_info_text.value]).to_csv(add_info_path, index=False, header=False)
        print(f"Additional information has been saved to {add_info_path}.")

    # Create and display the button
    button = widgets.Button(description="Save")
    button.on_click(on_button_click)
    display(button)

In [12]:
# Additional Information Example List
categories = {
    "style": [
        "Street", "Modern", "Classic", "Feminine", "Casual", "Sporty", "Vintage", "Elegant", "Minimal",
        "Bohemian", "Chic", "Preppy", "Grunge", "Punk", "Goth", "Romantic", "Resort", "Business"
    ],
    "color": [
        "Black", "White", "Red", "Blue", "Green", "Yellow", "Pink", "Gray", "Beige",
        "Brown", "Orange", "Purple", "Ivory", "Navy", "Khaki", "Mint", "Lavender", "Mustard", "Burgundy"
    ],
    "pattern": [
        "Solid", "Striped", "Checkered", "Floral", "Leopard", "Abstract",
        "Polka Dot", "Paisley", "Camouflage", "Houndstooth", "Animal Print", "Tie-Dye", "Geometric", "Embroidery"
    ],
    "occasion": [
        "Casual Brunch", "Formal Event", "Park Picnic", "Office Meeting",
        "Wedding Guest", "Date Night", "Beach Vacation", "Travel", "Daily Wear", "Birthday Party", "Business Casual"
    ],
    "season": [
        "Spring", "Summer", "Autumn", "Winter"
    ],
    "clothing": [
        "Outerwear", "Top", "Bottoms", "Dress"
    ],
    "length": [
        "Mini", "Midi", "Maxi",
        "Knee Length", "Ankle Length", "Cropped", "Hip Length", "Floor Length"
    ],
    "collar": [
        "Shirt Collar", "V-neck", "Round neck",
        "Turtleneck", "Collarless", "Boat Neck", "Square Neck", "Halter Neck", "Off-shoulder"
    ],
    "sleeve": [
        "Sleeveless", "Short Sleeve", "3/4 Sleeve", "Long Sleeve",
        "Balloon Sleeve", "Cap Sleeve", "Puff Sleeve", "Bishop Sleeve", "Dolman Sleeve"
    ],
    "fit": [
        "Loose", "Fitted", "Oversized",
        "Slim Fit", "Regular Fit", "Relaxed Fit", "Boxy", "Bodycon"
    ]
}

### Generate Instruction Dataset

In [13]:
Instruction_Dataset_Format = {"Prompt":
                                {"persona":"당신은 한국 20-30대 여성을 타겟으로 하는 패션 디자이너입니다.",
                                 "task_description": "Input과 Add_Info를 기반으로 여성이 옷을 입은 모습을 생성해주세요.",
                                 "constraint": "얼굴을 제외하고 여성의 전신이 나오도록 옷 입은 모습 이미지를 생성해주세요, 그림 형식이 아니라 사진 형식이여야 됩니다."},
                            "Input": "",
                            "Add_Info": "",
                            "Output": ""}

In [ ]:
def generate_instruction_dataset(Instruction_Dataset_Format, LABEL_FILE_LIST, file_index):
    # Load labeling data
    with open(CAPTION_FOLDER + LABEL_FILE_LIST[file_index], 'r', encoding='utf-8') as file:
        labeling = json.load(file)

    # Load source data image filename
    image = IMAGE_FILE[file_index]

    # Load additional information
    # add_info = pd.read_csv(ADD_INFO_FOLDER + os.path.basename(LABEL_FILE_LIST[file_index]).split('.')[0] + '.txt', header=None)[0][0]

    # Update Instruction_Dataset_Format
    Instruction_Dataset_Format['Input'] = labeling
    # Instruction_Dataset_Format['Add_Info'] = add_info
    Instruction_Dataset_Format['Output'] = image

    # Save Instruction_Dataset as a JSON file
    with open(INSTRUCTION_FOLDER + os.path.basename(LABEL_FILE_LIST[file_index]).split('.')[0] + '.json', 'w', encoding='utf-8') as file:
        json.dump(Instruction_Dataset_Format, file, ensure_ascii=False, indent=4)

    print("Instruction_Dataset:", Instruction_Dataset_Format)
    print('-' * 50)

for file_index in range(len(LABEL_FILE)):
    generate_instruction_dataset(Instruction_Dataset_Format, LABEL_FILE, file_index)

### Process Seed Label Data

In [15]:
for filename in os.listdir(SEED_LABEL_FOLDER):
    file_path = os.path.join(SEED_LABEL_FOLDER, filename)
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    data['Prompt'] = 'Create a full-body photo of a 20-30s Korean woman in the specified outfit, excluding the face.'
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

### Move Data

In [ ]:
COMBINED_IMAGE_FOLDER = config.get("COMBINED_IMAGE_KFASHION_FOLDER")
COMBINED_LABEL_FOLDER = config.get("COMBINED_LABEL_CLIP_FOLDER")

TRAIN_IMAGE_FOLDER = config.get("TRAIN_IMAGE_KFASHION_FOLDER")
TRAIN_LABEL_FOLDER = config.get("TRAIN_LABEL_CLIP_FOLDER")

TEST_IMAGE_FOLDER = config.get("TEST_IMAGE_KFASHION_FOLDER")
TEST_LABEL_FOLDER = config.get("TEST_LABEL_CLIP_FOLDER")

# Prepare file lists
image_files = {os.path.splitext(f)[0]: f for f in os.listdir(COMBINED_IMAGE_FOLDER) if f.endswith(".jpg")}
label_files = {os.path.splitext(f)[0]: f for f in os.listdir(COMBINED_LABEL_FOLDER) if f.endswith(".json")}

# Use only common prefixes
common_prefixes = sorted(set(image_files.keys()) & set(label_files.keys()))

# Split into training and testing sets
train_prefixes = common_prefixes[:2024]
test_prefixes = common_prefixes[2024:]

In [ ]:
# Copy function
def copy_pairs(prefixes, image_map, label_map, image_src, label_src, image_dst, label_dst):
    for prefix in prefixes:
        image_name = image_map[prefix]
        label_name = label_map[prefix]
        shutil.copy(os.path.join(image_src, image_name), os.path.join(image_dst, image_name))
        shutil.copy(os.path.join(label_src, label_name), os.path.join(label_dst, label_name))

# Perform copying
copy_pairs(train_prefixes, image_files, label_files, COMBINED_IMAGE_FOLDER, COMBINED_LABEL_FOLDER, TRAIN_IMAGE_FOLDER, TRAIN_LABEL_FOLDER)
copy_pairs(test_prefixes, image_files, label_files, COMBINED_IMAGE_FOLDER, COMBINED_LABEL_FOLDER, TEST_IMAGE_FOLDER, TEST_LABEL_FOLDER)

# Check train and test matching
train_images = sorted(os.listdir(TRAIN_IMAGE_FOLDER))
train_labels = sorted(os.listdir(TRAIN_LABEL_FOLDER))

for img_file, label_file in zip(train_images[:10], train_labels[:10]):
    assert os.path.splitext(img_file)[0] == os.path.splitext(label_file)[0], f"❌ Fail Matching: {img_file}, {label_file}"
print("✅ Train Pass Matching")

### Summarize

In [ ]:
def summarize_text(text, summarizer, max_token_len, min_token_len):
    result = summarizer(text, max_length=max_token_len, min_length=min_token_len, do_sample=False)
    summary_text = result[0]['summary_text']
    return summary_text
summarizer = pipeline("summarization", model=SUMMARIZE_MODEL, device=device)

In [ ]:
for filename in tqdm(os.listdir(COMBINED_LABEL_FOLDER)):
    file_path = os.path.join(COMBINED_LABEL_FOLDER, filename)
    
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    prompt = data.get('Prompt', '')
    input_caption = data.get('Input', {}).get('caption', '')
    add_info = data.get('Add_Info', '')

    full_text = " ".join([prompt, input_caption, add_info])
    summary_text = summarize_text(full_text, summarizer, max_token_len=80, min_token_len=30)

    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump({"Summary": summary_text}, f, ensure_ascii=False, indent=4)